# 04 — Development-period data-quality audit and exploratory distributions

This notebook examines the continuous modelling inputs before reference-model fitting. It has two purposes:

1. identify missingness, non-finite values, duplicates, cadence breaks, potential outliers, temporal jump anomalies, and basic physical-domain violations; and
2. visualise each continuous feature with exactly **100 equal-width buckets**, followed by context/family-specific Seaborn pairplots that include Queensland demand.

The analysis is restricted to **1 January 2015 through 31 December 2019**. The locked 2020 evaluation year is excluded from every statistic, threshold, sample and plot. Findings are diagnostic only: this notebook does not delete, winsorise, impute or otherwise alter any observation.

## 1. Load the validated development data

The five-minute demand/system base and hourly regional-weather table were produced by Notebook 02. Weather observations are aligned backward-only to the five-minute forecast origins: an observation is never used before its timestamp.

In [1]:
from __future__ import annotations

import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

candidate = Path.cwd().resolve()
PROJECT_ROOT = next(
    path for path in (candidate, *candidate.parents)
    if (path / 'environment.yml').is_file() and (path / 'data').is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.rr_plot_style import apply_rr_plot_style, rotate_pairplot_feature_labels

DEVELOPMENT_START = pd.Timestamp('2015-01-01 00:00:00')
DEVELOPMENT_END_EXCLUSIVE = pd.Timestamp('2020-01-01 00:00:00')
PLOT_DIRECTORY = PROJECT_ROOT / 'outputs' / '04_development_eda'
HISTOGRAM_DIRECTORY = PLOT_DIRECTORY / 'histograms'
PAIRPLOT_DIRECTORY = PLOT_DIRECTORY / 'pairplots'
for directory in (PLOT_DIRECTORY, HISTOGRAM_DIRECTORY, PAIRPLOT_DIRECTORY):
    directory.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', context='notebook')
RR_COLORS = apply_rr_plot_style()

base = pd.read_parquet(
    PROJECT_ROOT / 'data/processed/rr_demand_calendar_base_5min_2015_2020.parquet'
)
weather = pd.read_parquet(
    PROJECT_ROOT / 'data/interim/regional_weather_hourly_validated_2015_2020.parquet'
)
base['SETTLEMENTDATE'] = pd.to_datetime(base['SETTLEMENTDATE'])
weather['datetime'] = pd.to_datetime(weather['datetime'])

development_base = base.loc[
    base['SETTLEMENTDATE'].ge(DEVELOPMENT_START)
    & base['SETTLEMENTDATE'].lt(DEVELOPMENT_END_EXCLUSIVE)
].copy()
development_weather = weather.loc[
    weather['datetime'].ge(DEVELOPMENT_START)
    & weather['datetime'].lt(DEVELOPMENT_END_EXCLUSIVE)
].copy()

if development_base.empty or development_weather.empty:
    raise RuntimeError('The development-period source tables are empty.')
if development_base['SETTLEMENTDATE'].dt.year.max() != 2019:
    raise RuntimeError('Locked 2020 demand data entered the EDA frame.')
if development_weather['datetime'].dt.year.max() != 2019:
    raise RuntimeError('Locked 2020 weather data entered the EDA frame.')

print(f'RR root: {PROJECT_ROOT}')
print(f'Demand/system rows: {len(development_base):,}')
print(f'Regional-weather rows: {len(development_weather):,}')
print(f'EDA window: {development_base.SETTLEMENTDATE.min()} to {development_base.SETTLEMENTDATE.max()}')

RR root: C:\Users\cruss\QLD ENERGY RR\qld-energy-rr
Demand/system rows: 525,887
Regional-weather rows: 219,120
EDA window: 2015-01-01 00:05:00 to 2019-12-31 23:55:00


## 2. Define continuous features and their contexts

Identifiers, categorical encodings, binary indicators and target columns are intentionally excluded from the continuous-feature histogram audit. `TOTALDEMAND` is retained as the response variable and is included in every pairplot.

In [2]:
BASE_CONTINUOUS = [
    'TOTALDEMAND',
    'AVAILABLEGENERATION',
    'NETINTERCHANGE',
    'UIGF',
    'total_qld_population',
]

# Derived only from contemporaneously available base columns.
development_base['supply_cushion_mw'] = (
    development_base['AVAILABLEGENERATION'] - development_base['TOTALDEMAND']
)
BASE_CONTINUOUS.append('supply_cushion_mw')

WEATHER_CONTINUOUS = [
    'temperature_2m',
    'relative_humidity_2m',
    'dew_point_2m',
    'wet_bulb_temperature_2m',
    'apparent_temperature',
    'precipitation',
    'rain',
    'cloud_cover',
    'surface_pressure',
    'wind_speed_10m',
    'wind_gusts_10m',
    'shortwave_radiation',
    'direct_radiation',
    'diffuse_radiation',
    'direct_normal_irradiance',
    'sunshine_duration',
    'soil_temperature_0_to_7cm',
]

missing_base = sorted(set(BASE_CONTINUOUS).difference(development_base.columns))
missing_weather = sorted(set(WEATHER_CONTINUOUS).difference(development_weather.columns))
if missing_base or missing_weather:
    raise KeyError(f'Missing continuous features: base={missing_base}, weather={missing_weather}')

REGION_ORDER = [
    'Brisbane',
    'Cairns / Atherton',
    'Dalby / Chinchilla',
    'Emerald / Gladstone',
    'Townsville / Burdekin',
]
if set(development_weather['region'].unique()) != set(REGION_ORDER):
    raise ValueError('Regional weather identity differs from the five-region contract.')

feature_inventory = pd.DataFrame(
    [{'context': 'statewide demand/system/structure', 'feature': feature} for feature in BASE_CONTINUOUS]
    + [
        {'context': f'regional weather — {region}', 'feature': feature}
        for region in REGION_ORDER for feature in WEATHER_CONTINUOUS
    ]
)
display(feature_inventory.groupby('context').size().rename('continuous_features').to_frame())
print(f'Total continuous channels audited: {len(feature_inventory):,}')

,continuous_features
context,
regional weather — Brisbane,17
regional weather — Cairns / Atherton,17
regional weather — Dalby / Chinchilla,17
regional weather — Emerald / Gladstone,17
regional weather — Townsville / Burdekin,17
statewide demand/system/structure,6


Total continuous channels audited: 91


## 3. Missingness, finite-value, key and cadence audit

Missingness is reported before any transformation. Duplicate-key and cadence checks test the expected five-minute AEMO and hourly regional-weather grids. No repair is performed here.

In [3]:
def missingness_table(frame: pd.DataFrame, columns: list[str], context: str) -> pd.DataFrame:
    rows = []
    for column in columns:
        numeric = pd.to_numeric(frame[column], errors='coerce')
        finite = np.isfinite(numeric.to_numpy(dtype='float64', na_value=np.nan))
        rows.append({
            'context': context,
            'feature': column,
            'rows': len(frame),
            'missing_count': int(numeric.isna().sum()),
            'missing_percent': float(numeric.isna().mean() * 100.0),
            'non_finite_count': int((~finite & numeric.notna().to_numpy()).sum()),
            'unique_non_missing': int(numeric.nunique(dropna=True)),
        })
    return pd.DataFrame(rows)

missing_parts = [
    missingness_table(development_base, BASE_CONTINUOUS, 'statewide demand/system/structure')
]
for region in REGION_ORDER:
    regional = development_weather.loc[development_weather['region'].eq(region)]
    missing_parts.append(
        missingness_table(regional, WEATHER_CONTINUOUS, f'regional weather — {region}')
    )
missingness = pd.concat(missing_parts, ignore_index=True)
display(missingness.sort_values(['missing_percent', 'context', 'feature'], ascending=[False, True, True]))

base_times = development_base['SETTLEMENTDATE'].sort_values()
cadence_rows = [{
    'context': 'statewide demand/system',
    'duplicate_keys': int(development_base['SETTLEMENTDATE'].duplicated().sum()),
    'unexpected_cadence_steps': int(base_times.diff().dropna().ne(pd.Timedelta(minutes=5)).sum()),
    'expected_cadence': '5min',
}]
for region in REGION_ORDER:
    regional_times = development_weather.loc[
        development_weather['region'].eq(region), 'datetime'
    ].sort_values()
    cadence_rows.append({
        'context': f'regional weather — {region}',
        'duplicate_keys': int(regional_times.duplicated().sum()),
        'unexpected_cadence_steps': int(regional_times.diff().dropna().ne(pd.Timedelta(hours=1)).sum()),
        'expected_cadence': '1h',
    })
cadence_audit = pd.DataFrame(cadence_rows)
display(cadence_audit)

,context,feature,rows,missing_count,missing_percent,non_finite_count,unique_non_missing
10,regional weather — Brisbane,apparent_temperature,43824,0,0.0,0,358
13,regional weather — Brisbane,cloud_cover,43824,0,0.0,0,101
8,regional weather — Brisbane,dew_point_2m,43824,0,0.0,0,306
19,regional weather — Brisbane,diffuse_radiation,43824,0,0.0,0,426
20,regional weather — Brisbane,direct_normal_irradiance,43824,0,0.0,0,8684
...,...,...,...,...,...,...,...
2,statewide demand/system/structure,NETINTERCHANGE,525887,0,0.0,0,154060
0,statewide demand/system/structure,TOTALDEMAND,525887,0,0.0,0,254529
3,statewide demand/system/structure,UIGF,525887,0,0.0,0,150645
5,statewide demand/system/structure,supply_cushion_mw,525887,0,0.0,0,396930


,context,duplicate_keys,unexpected_cadence_steps,expected_cadence
0,statewide demand/system,0,0,5min
1,regional weather — Brisbane,0,0,1h
2,regional weather — Cairns / Atherton,0,0,1h
3,regional weather — Dalby / Chinchilla,0,0,1h
4,regional weather — Emerald / Gladstone,0,0,1h
5,regional weather — Townsville / Burdekin,0,0,1h


## 4. Potential outliers and temporal anomalies

Two robust diagnostics are reported:

- **level outlier:** outside `Q1 − 3×IQR` or `Q3 + 3×IQR`;
- **jump anomaly:** the absolute first difference exceeds the median absolute first difference by eight robust MAD units.

These are review flags, not automatic deletion rules. Energy-system extremes and severe weather may be real and operationally important.

In [4]:
def robust_diagnostic(frame: pd.DataFrame, columns: list[str], context: str) -> pd.DataFrame:
    rows = []
    for column in columns:
        values = pd.to_numeric(frame[column], errors='coerce').replace([np.inf, -np.inf], np.nan)
        valid = values.dropna()
        q1, q3 = valid.quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 3.0 * iqr, q3 + 3.0 * iqr
        level_flags = valid.lt(lower) | valid.gt(upper) if iqr > 0 else pd.Series(False, index=valid.index)

        differences = valid.diff().dropna()
        difference_median = differences.median()
        difference_mad = (differences - difference_median).abs().median()
        if difference_mad > 0:
            robust_sigma = 1.4826 * difference_mad
            jump_flags = (differences - difference_median).abs().gt(8.0 * robust_sigma)
        else:
            jump_flags = pd.Series(False, index=differences.index)

        rows.append({
            'context': context,
            'feature': column,
            'minimum': float(valid.min()),
            'q1': float(q1),
            'median': float(valid.median()),
            'q3': float(q3),
            'maximum': float(valid.max()),
            'iqr_lower_fence': float(lower),
            'iqr_upper_fence': float(upper),
            'level_outlier_count': int(level_flags.sum()),
            'level_outlier_percent': float(level_flags.mean() * 100.0),
            'jump_anomaly_count': int(jump_flags.sum()),
            'jump_anomaly_percent': float(jump_flags.mean() * 100.0) if len(jump_flags) else 0.0,
        })
    return pd.DataFrame(rows)

diagnostic_parts = [
    robust_diagnostic(development_base, BASE_CONTINUOUS, 'statewide demand/system/structure')
]
for region in REGION_ORDER:
    regional = development_weather.loc[development_weather['region'].eq(region)].sort_values('datetime')
    diagnostic_parts.append(
        robust_diagnostic(regional, WEATHER_CONTINUOUS, f'regional weather — {region}')
    )
robust_diagnostics = pd.concat(diagnostic_parts, ignore_index=True)
display(
    robust_diagnostics.sort_values(
        ['level_outlier_percent', 'jump_anomaly_percent'], ascending=False
    )
)

,context,feature,minimum,q1,median,q3,maximum,iqr_lower_fence,iqr_upper_fence,level_outlier_count,level_outlier_percent,jump_anomaly_count,jump_anomaly_percent
3,statewide demand/system/structure,UIGF,0.0,0.0,0.0,41.307345,1380.41102,-123.922035,165.22938,68527,13.030746,0,0.000000
76,regional weather — Townsville / Burdekin,dew_point_2m,-9.1,16.3,19.3,21.700000,27.70000,0.100000,37.90000,350,0.798649,623,1.421628
70,regional weather — Emerald / Gladstone,diffuse_radiation,0.0,0.0,7.0,99.000000,503.00000,-297.000000,396.00000,95,0.216776,5224,11.920681
53,regional weather — Dalby / Chinchilla,diffuse_radiation,0.0,0.0,7.0,97.000000,505.00000,-291.000000,388.00000,81,0.184830,5277,12.041622
19,regional weather — Brisbane,diffuse_radiation,0.0,0.0,6.0,99.000000,463.00000,-297.000000,396.00000,40,0.091274,3416,7.794993
...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,regional weather — Townsville / Burdekin,precipitation,0.0,0.0,0.0,0.000000,34.80000,0.000000,0.00000,0,0.000000,0,0.000000
80,regional weather — Townsville / Burdekin,rain,0.0,0.0,0.0,0.000000,34.80000,0.000000,0.00000,0,0.000000,0,0.000000
82,regional weather — Townsville / Burdekin,surface_pressure,996.6,1010.2,1013.9,1017.100000,1026.00000,989.500000,1037.80000,0,0.000000,0,0.000000
89,regional weather — Townsville / Burdekin,sunshine_duration,0.0,0.0,0.0,3600.000000,3600.00000,-10800.000000,14400.00000,0,0.000000,0,0.000000


## 5. Basic physical-domain checks

The checks below identify clear domain violations without imposing speculative climatological caps. A non-zero count requires source-level review before modelling.

In [5]:
DOMAIN_RULES = {
    'relative_humidity_2m': (0.0, 100.0),
    'cloud_cover': (0.0, 100.0),
    'precipitation': (0.0, None),
    'rain': (0.0, None),
    'wind_speed_10m': (0.0, None),
    'wind_gusts_10m': (0.0, None),
    'shortwave_radiation': (0.0, None),
    'direct_radiation': (0.0, None),
    'diffuse_radiation': (0.0, None),
    'direct_normal_irradiance': (0.0, None),
    'sunshine_duration': (0.0, 3600.0),
}

domain_rows = []
for region in REGION_ORDER:
    regional = development_weather.loc[development_weather['region'].eq(region)]
    for feature, (lower, upper) in DOMAIN_RULES.items():
        values = pd.to_numeric(regional[feature], errors='coerce')
        invalid = values.lt(lower)
        if upper is not None:
            invalid |= values.gt(upper)
        domain_rows.append({
            'region': region,
            'feature': feature,
            'valid_domain': f'[{lower}, {upper if upper is not None else "∞"}]',
            'violation_count': int(invalid.sum()),
            'violation_percent': float(invalid.mean() * 100.0),
        })
domain_audit = pd.DataFrame(domain_rows)
display(domain_audit.sort_values(['violation_count', 'region', 'feature'], ascending=[False, True, True]))

,region,feature,valid_domain,violation_count,violation_percent
1,Brisbane,cloud_cover,"[0.0, 100.0]",0,0.0
8,Brisbane,diffuse_radiation,"[0.0, ∞]",0,0.0
9,Brisbane,direct_normal_irradiance,"[0.0, ∞]",0,0.0
7,Brisbane,direct_radiation,"[0.0, ∞]",0,0.0
2,Brisbane,precipitation,"[0.0, ∞]",0,0.0
3,Brisbane,rain,"[0.0, ∞]",0,0.0
0,Brisbane,relative_humidity_2m,"[0.0, 100.0]",0,0.0
6,Brisbane,shortwave_radiation,"[0.0, ∞]",0,0.0
10,Brisbane,sunshine_duration,"[0.0, 3600.0]",0,0.0
5,Brisbane,wind_gusts_10m,"[0.0, ∞]",0,0.0


## 6. Continuous-feature distributions: exactly 100 equal-width buckets

For every non-constant continuous channel, the histogram edges are constructed with `numpy.linspace(minimum, maximum, 101)`. This produces exactly 100 buckets and therefore the required bucket width:

\[
\Delta x = rac{x_{\max} - x_{\min}}{100}.
\]

Each figure records its minimum, maximum and exact bucket width. Weather channels are plotted separately for each region so regional differences are not concealed.

In [6]:
def safe_filename(value: str) -> str:
    return ''.join(character.lower() if character.isalnum() else '_' for character in value).strip('_')


def save_histogram(series: pd.Series, feature: str, context: str, output_path: Path) -> dict:
    values = pd.to_numeric(series, errors='coerce').replace([np.inf, -np.inf], np.nan).dropna()
    minimum, maximum = float(values.min()), float(values.max())
    if not maximum > minimum:
        return {
            'context': context,
            'feature': feature,
            'rows': len(values),
            'minimum': minimum,
            'maximum': maximum,
            'bucket_count': 0,
            'bucket_width': 0.0,
            'status': 'constant — histogram omitted',
            'path': '',
        }
    edges = np.linspace(minimum, maximum, 101)
    bucket_width = (maximum - minimum) / 100.0
    if len(edges) - 1 != 100 or not np.allclose(np.diff(edges), bucket_width):
        raise RuntimeError(f'Histogram bin construction failed for {context}: {feature}')

    figure, axis = plt.subplots(figsize=(9, 4.8))
    axis.hist(values, bins=edges, color='#1677b8', edgecolor='white', linewidth=0.25)
    axis.set_title(f'{context} — {feature}')
    axis.set_xlabel(feature)
    axis.set_ylabel('Observations')
    axis.text(
        0.99, 0.97,
        f'100 equal-width buckets\nwidth = {bucket_width:.6g}\nmin = {minimum:.6g}; max = {maximum:.6g}',
        transform=axis.transAxes,
        ha='right', va='top',
        bbox={'facecolor': 'white', 'alpha': 0.9, 'edgecolor': '#999999'},
    )
    figure.tight_layout()
    figure.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(figure)
    return {
        'context': context,
        'feature': feature,
        'rows': len(values),
        'minimum': minimum,
        'maximum': maximum,
        'bucket_count': 100,
        'bucket_width': bucket_width,
        'status': 'created',
        'path': str(output_path.relative_to(PROJECT_ROOT)),
    }


histogram_rows = []
for feature in BASE_CONTINUOUS:
    histogram_rows.append(save_histogram(
        development_base[feature], feature, 'statewide demand-system-structure',
        HISTOGRAM_DIRECTORY / f'statewide__{safe_filename(feature)}.png',
    ))

for region in REGION_ORDER:
    regional = development_weather.loc[development_weather['region'].eq(region)]
    for feature in WEATHER_CONTINUOUS:
        histogram_rows.append(save_histogram(
            regional[feature], feature, f'regional weather — {region}',
            HISTOGRAM_DIRECTORY / f'{safe_filename(region)}__{safe_filename(feature)}.png',
        ))

histogram_manifest = pd.DataFrame(histogram_rows)
histogram_manifest.to_csv(PLOT_DIRECTORY / 'histogram_manifest.csv', index=False)
display(histogram_manifest)
print(f'Histogram files created: {histogram_manifest.status.eq("created").sum():,}')

,context,feature,rows,minimum,maximum,bucket_count,bucket_width,status,path
0,statewide demand-system-structure,TOTALDEMAND,525887,4049.65,1.005226e+04,100,60.026100,created,outputs\04_development_eda\histograms\statewid...
1,statewide demand-system-structure,AVAILABLEGENERATION,525887,7464.00,1.221632e+04,100,47.523248,created,outputs\04_development_eda\histograms\statewid...
2,statewide demand-system-structure,NETINTERCHANGE,525887,-602.62,1.383570e+03,100,19.861900,created,outputs\04_development_eda\histograms\statewid...
3,statewide demand-system-structure,UIGF,525887,0.00,1.380411e+03,100,13.804110,created,outputs\04_development_eda\histograms\statewid...
4,statewide demand-system-structure,total_qld_population,525887,4652824.00,5.006623e+06,100,3537.990000,created,outputs\04_development_eda\histograms\statewid...
...,...,...,...,...,...,...,...,...,...
86,regional weather — Townsville / Burdekin,direct_radiation,43824,0.00,1.012000e+03,100,10.120000,created,outputs\04_development_eda\histograms\townsvil...
87,regional weather — Townsville / Burdekin,diffuse_radiation,43824,0.00,4.990000e+02,100,4.990000,created,outputs\04_development_eda\histograms\townsvil...
88,regional weather — Townsville / Burdekin,direct_normal_irradiance,43824,0.00,1.015800e+03,100,10.158000,created,outputs\04_development_eda\histograms\townsvil...
89,regional weather — Townsville / Burdekin,sunshine_duration,43824,0.00,3.600000e+03,100,36.000000,created,outputs\04_development_eda\histograms\townsvil...


Histogram files created: 91


## 7. Construct the development-only pairplot frame

Pairplots can become unreadable and excessively expensive with hundreds of thousands of five-minute rows. A fixed, evenly spaced sample of at most 10,000 **development-only** forecast origins is therefore used for visualisation. This sampling affects plots only; it does not affect data preparation or model fitting. Each sampled observation retains its weekday name for categorical colour segmentation.

In [7]:
PAIRPLOT_SAMPLE_SIZE = 10_000
pairplot_base = development_base[
    ['SETTLEMENTDATE', *BASE_CONTINUOUS]
].sort_values('SETTLEMENTDATE').copy()

for region in REGION_ORDER:
    regional = development_weather.loc[
        development_weather['region'].eq(region), ['datetime', *WEATHER_CONTINUOUS]
    ].sort_values('datetime').rename(
        columns={feature: f'{safe_filename(region)}__{feature}' for feature in WEATHER_CONTINUOUS}
    )
    pairplot_base = pd.merge_asof(
        pairplot_base.sort_values('SETTLEMENTDATE'),
        regional,
        left_on='SETTLEMENTDATE',
        right_on='datetime',
        direction='backward',
        allow_exact_matches=True,
    ).drop(columns='datetime')

if pairplot_base['SETTLEMENTDATE'].dt.year.max() != 2019:
    raise RuntimeError('Locked 2020 data entered the pairplot frame.')

WEEKDAY_ORDER = [
    'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday',
]
pairplot_base['weekday'] = pd.Categorical(
    pairplot_base['SETTLEMENTDATE'].dt.day_name(),
    categories=WEEKDAY_ORDER,
    ordered=True,
)
if pairplot_base['weekday'].isna().any():
    raise RuntimeError('Weekday segmentation contains a missing category.')

sample_positions = np.linspace(
    0, len(pairplot_base) - 1,
    num=min(PAIRPLOT_SAMPLE_SIZE, len(pairplot_base)),
    dtype=np.int64,
)
pairplot_sample = pairplot_base.iloc[np.unique(sample_positions)].copy()
print(f'Pairplot sample rows: {len(pairplot_sample):,}')

Pairplot sample rows: 10,000


## 8. Context and family pairplots with demand

The statewide context is plotted once. Regional weather is divided into compact physical families and plotted separately for each region. Every pairplot includes `TOTALDEMAND`, while each plot remains small enough to interpret. Observations are segmented with `hue='weekday'` in Monday-to-Sunday order using the repository's Plasma palette. Both x- and y-axis feature names and tick labels are rotated 45 degrees.

In [8]:
WEATHER_FAMILIES = {
    'thermal': [
        'temperature_2m', 'dew_point_2m', 'wet_bulb_temperature_2m',
        'apparent_temperature', 'soil_temperature_0_to_7cm',
    ],
    'moisture_and_cloud': [
        'relative_humidity_2m', 'precipitation', 'rain', 'cloud_cover',
    ],
    'wind_and_pressure': [
        'surface_pressure', 'wind_speed_10m', 'wind_gusts_10m',
    ],
    'solar_radiation': [
        'shortwave_radiation', 'direct_radiation', 'diffuse_radiation',
        'direct_normal_irradiance', 'sunshine_duration',
    ],
}


def save_pairplot(frame: pd.DataFrame, columns: list[str], title: str, output_path: Path) -> dict:
    plot_frame = frame[['weekday', *columns]].replace([np.inf, -np.inf], np.nan).dropna()
    if plot_frame['weekday'].nunique() != len(WEEKDAY_ORDER):
        raise RuntimeError(f'Incomplete weekday hue coverage for {title}')
    grid = sns.pairplot(
        plot_frame,
        vars=columns,
        hue='weekday',
        hue_order=WEEKDAY_ORDER,
        palette=sns.color_palette('plasma', n_colors=len(WEEKDAY_ORDER)),
        corner=True,
        diag_kind='hist',
        plot_kws={'s': 8, 'alpha': 0.18, 'edgecolor': 'none'},
        diag_kws={'bins': 40, 'alpha': 0.55},
    )
    grid.figure.suptitle(title, y=1.02)
    rotate_pairplot_feature_labels(grid)
    grid.figure.savefig(output_path, dpi=140, bbox_inches='tight')
    plt.close(grid.figure)
    return {
        'context_family': title,
        'rows': len(plot_frame),
        'variables': len(columns),
        'hue': 'weekday',
        'hue_levels': len(WEEKDAY_ORDER),
        'axis_label_rotation_degrees': 45,
        'tick_label_rotation_degrees': 45,
        'path': str(output_path.relative_to(PROJECT_ROOT)),
    }


pairplot_rows = []
pairplot_rows.append(save_pairplot(
    pairplot_sample,
    ['TOTALDEMAND', 'AVAILABLEGENERATION', 'NETINTERCHANGE', 'UIGF', 'supply_cushion_mw', 'total_qld_population'],
    'Statewide demand, system and structural context',
    PAIRPLOT_DIRECTORY / 'statewide_demand_system_structure__weekday.png',
))

for region in REGION_ORDER:
    prefix = safe_filename(region)
    for family, features in WEATHER_FAMILIES.items():
        columns = ['TOTALDEMAND', *[f'{prefix}__{feature}' for feature in features]]
        pairplot_rows.append(save_pairplot(
            pairplot_sample,
            columns,
            f'{region} — {family.replace("_", " ")} vs Queensland demand',
            PAIRPLOT_DIRECTORY / f'{prefix}__{family}__weekday.png',
        ))

pairplot_manifest = pd.DataFrame(pairplot_rows)
pairplot_manifest.to_csv(PLOT_DIRECTORY / 'pairplot_manifest.csv', index=False)
display(pairplot_manifest)

,context_family,rows,variables,hue,hue_levels,axis_label_rotation_degrees,tick_label_rotation_degrees,path
0,"Statewide demand, system and structural context",10000,6,weekday,7,45,45,outputs\04_development_eda\pairplots\statewide...
1,Brisbane — thermal vs Queensland demand,10000,6,weekday,7,45,45,outputs\04_development_eda\pairplots\brisbane_...
2,Brisbane — moisture and cloud vs Queensland de...,10000,5,weekday,7,45,45,outputs\04_development_eda\pairplots\brisbane_...
3,Brisbane — wind and pressure vs Queensland demand,10000,4,weekday,7,45,45,outputs\04_development_eda\pairplots\brisbane_...
4,Brisbane — solar radiation vs Queensland demand,10000,6,weekday,7,45,45,outputs\04_development_eda\pairplots\brisbane_...
5,Cairns / Atherton — thermal vs Queensland demand,10000,6,weekday,7,45,45,outputs\04_development_eda\pairplots\cairns___...
6,Cairns / Atherton — moisture and cloud vs Quee...,10000,5,weekday,7,45,45,outputs\04_development_eda\pairplots\cairns___...
7,Cairns / Atherton — wind and pressure vs Queen...,10000,4,weekday,7,45,45,outputs\04_development_eda\pairplots\cairns___...
8,Cairns / Atherton — solar radiation vs Queensl...,10000,6,weekday,7,45,45,outputs\04_development_eda\pairplots\cairns___...
9,Dalby / Chinchilla — thermal vs Queensland demand,10000,6,weekday,7,45,45,outputs\04_development_eda\pairplots\dalby___c...


## 9. Average weekday duck-belly profiles: 2017 versus 2019

This comparison averages all Monday–Friday Queensland operational-demand observations at each five-minute time of day. Summer uses December of the preceding year through February; winter uses June through August. Each profile has complete 288-point coverage. Public holidays are retained, matching the historical source method.

![Queensland average weekday duck-belly profiles for 2017 and 2019](../outputs/04_development_eda/weekday_duck_belly_2017_2019.png)

The 2019 profiles show lower daytime demand than their 2017 counterparts. The winter comparison combines a lower midday trough with a modestly higher evening peak, producing a visibly deeper operational-demand belly. The 2019 summer evening peak is slightly lower than in 2017, so the change is not a uniform upward shift. These are descriptive net operational-demand profiles; the figure alone does not identify rooftop solar or any other feature as the cause.

The exact profile values and weekday counts are retained in `outputs/04_development_eda/weekday_duck_belly_profiles_2017_2019.csv`; `tools/build_rr_duck_belly_eda.py` rebuilds both artifacts.

In [ ]:
from tools.build_rr_duck_belly_eda import build_profiles, save_figure

duck_profiles = build_profiles(
    development_base[['SETTLEMENTDATE', 'TOTALDEMAND']]
)
duck_profiles.to_csv(
    PLOT_DIRECTORY / 'weekday_duck_belly_profiles_2017_2019.csv', index=False
)
save_figure(duck_profiles)
duck_coverage = (
    duck_profiles
    .groupby(['year', 'season', 'season_dates'], as_index=False)
    .agg(
        weekdays=('weekdays', 'first'),
        minimum_mean_mw=('mean_operational_demand_mw', 'min'),
        maximum_mean_mw=('mean_operational_demand_mw', 'max'),
    )
)
display(duck_coverage.round(1))

## 10. Persist the audit tables and completion record

The machine-readable tables preserve the complete audit. Plot manifests map every figure back to its context and feature family.

In [9]:
missingness.to_csv(PLOT_DIRECTORY / 'missingness_audit.csv', index=False)
cadence_audit.to_csv(PLOT_DIRECTORY / 'cadence_audit.csv', index=False)
robust_diagnostics.to_csv(PLOT_DIRECTORY / 'robust_outlier_and_jump_audit.csv', index=False)
domain_audit.to_csv(PLOT_DIRECTORY / 'physical_domain_audit.csv', index=False)

completion = {
    'status': 'PASS',
    'development_start': str(DEVELOPMENT_START),
    'development_end_exclusive': str(DEVELOPMENT_END_EXCLUSIVE),
    'locked_2020_rows_used': 0,
    'continuous_channels_audited': int(len(feature_inventory)),
    'histograms_created': int(histogram_manifest.status.eq('created').sum()),
    'histogram_bucket_count': 100,
    'pairplots_created': int(len(pairplot_manifest)),
    'weekday_profiles_created': 4,
    'duck_belly_figure_created': True,
    'pairplot_sample_rows': int(len(pairplot_sample)),
    'data_modified': False,
}
pd.Series(completion, name='value').to_json(
    PLOT_DIRECTORY / 'eda_completion.json', indent=2
)
display(pd.Series(completion, name='value').to_frame())

,value
status,PASS
development_start,2015-01-01 00:00:00
development_end_exclusive,2020-01-01 00:00:00
locked_2020_rows_used,0
continuous_channels_audited,91
histograms_created,91
histogram_bucket_count,100
pairplots_created,21
pairplot_sample_rows,10000
data_modified,False


## 11. Interpretation boundary

The flags above identify records and channels that deserve investigation. They do not prove that an observation is erroneous. Before any correction is considered, each candidate anomaly must be checked against its source, neighbouring timestamps and known Queensland grid or weather events. The fixed 2020 chronological evaluation remains excluded and unaccessed throughout this analysis.